In [ ]:
%pip install lightning adversarial-robustness-toolbox[pytorch_image] umap-learn tsnecuda==3.0.1+cu122 -f https://tsnecuda.isx.ai/tsnecuda_stable.html

# Libraries

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import absolute_import, division, print_function, unicode_literals, annotations
import os
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Callable, TYPE_CHECKING
from dataclasses import dataclass

import h5py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.metrics import *

from tqdm.notebook import tqdm
from joblib import delayed, Parallel
from scipy.optimize import minimize

import kagglehub

from einops import rearrange
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import transforms
import torchvision.models as models
from torchvision.transforms.functional import gaussian_blur
import torchmetrics

import art
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent
from art.estimators.classification import PyTorchClassifier
from art.config import ART_NUMPY_DTYPE
from art.defences.preprocessor import SpatialSmoothingPyTorch

if TYPE_CHECKING:
    from art.utils import CLIP_VALUES_TYPE

from lightning import LightningModule

# Dataset

In [ ]:
class PCAMDataset(Dataset):
    """Optimized PCAM Dataset with lazy loading"""
    
    def __init__(self, input_file_path: str, label_file_path: str, 
                 transform=None, max_samples: Optional[int] = None):
        self.input_file_path = input_file_path
        self.label_file_path = label_file_path
        self.transform = transform
        self.max_samples = max_samples
        
        # Open files and keep references
        self.input_file = h5py.File(self.input_file_path, 'r')
        self.label_file = h5py.File(self.label_file_path, 'r')
        
        self.input_data = self.input_file["x"]
        self.label_data = self.label_file["y"]
        
        # Set length
        self.length = len(self.input_data)
        if max_samples is not None:
            self.length = min(max_samples, self.length)
            print(f"Using {max_samples}/{len(self.input_data)} samples")
            
    def __len__(self) -> int:
        return self.length
    
    def __getitem__(self, idx: int):
        if idx >= self.length:
            raise IndexError("Index out of range")
        
        # Load image and convert to PIL
        image = Image.fromarray(self.input_data[idx]).convert("RGB")
        label = int(self.label_data[idx, 0, 0, 0])
        
        if self.transform:
            image = self.transform(image)
            
        return image, label
    
    def __del__(self):
        # Close files when dataset is destroyed
        if hasattr(self, 'input_file'):
            self.input_file.close()
        if hasattr(self, 'label_file'):
            self.label_file.close()

# ResNet18 Classifier

In [ ]:
class ResNet18Classifier(LightningModule):
    def __init__(self, num_classes: int = 2, label_smoothing: float = 0.0):
        super().__init__()
        self.model = models.resnet18(weights='DEFAULT')
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
        self.num_classes = num_classes
        
        self.criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

        self.train_acc = torchmetrics.Accuracy("binary", num_classes=num_classes)
        self.train_auroc = torchmetrics.AUROC("binary", num_classes=num_classes)
        self.val_acc = torchmetrics.Accuracy("binary", num_classes=num_classes)
        self.val_auroc = torchmetrics.AUROC("binary", num_classes=num_classes)
        self.test_acc = torchmetrics.Accuracy("binary", num_classes=num_classes)
        self.test_auroc = torchmetrics.AUROC("binary", num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

    def _step(self, batch, stage: str):
        images, labels = batch

        outputs = self(images)
        loss = self.criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1)
        probs = F.softmax(outputs, dim=1)

        if stage == "train":
            self.train_acc(preds, labels)
            self.train_auroc(probs, labels)
        elif stage == "val":
            self.val_acc(preds, labels)
            self.val_auroc(probs, labels)
        elif stage == "test":
            self.test_acc(preds, labels)
            self.test_auroc(probs, labels)
        else:
            raise ValueError(f"Unknown stage: {stage}")
        
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)        
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")
    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")
    def test_step(self, batch, batch_idx):
        return self._step(batch, "test")

    def on_train_epoch_end(self):
        self.log("train_acc", self.train_acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_auroc", self.train_auroc, on_step=False, on_epoch=True, prog_bar=True)
        self.train_acc.reset()
        self.train_auroc.reset()
    
    def on_validation_epoch_end(self):
        self.log("val_acc", self.val_acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_auroc", self.val_auroc, on_step=False, on_epoch=True, prog_bar=True)
        self.val_acc.reset()
        self.val_auroc.reset()

    def on_test_epoch_end(self):
        self.log("test_acc", self.test_acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log("test_auroc", self.test_auroc, on_step=False, on_epoch=True, prog_bar=True)
        self.test_acc.reset()
        self.test_auroc.reset()

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=1e-3, weight_decay=0.05)
        
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.1, patience=5
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1,
            },
        }
        
    def freeze_backbone(self):
        for param in self.model.parameters():
            param.requires_grad = False
        for param in self.model.fc.parameters():  # Keep classifier trainable
            param.requires_grad = True

    def unfreeze_backbone(self):
        for param in self.model.parameters():
            param.requires_grad = True

## Configuration

In [ ]:
DATA_ROOT = kagglehub.dataset_download("andrewmvd/metastatic-tissue-classification-patchcamelyon")
print("Path to dataset files:", DATA_ROOT)

In [ ]:
MODEL_PATH = kagglehub.model_download("electroduck/hermes_resnet_18/pyTorch/lightning")
print("Path to model files:", MODEL_PATH)

In [ ]:
# There are two model checkpoints available:
# 1. without data augmentation: https://www.kaggle.com/models/electroduck/hermes_resnet_18/pyTorch/no_augmentation_lightning/1/reduced_dataset_no_data_augmentation.ckpt
# 2. with data augmentation: https://www.kaggle.com/models/electroduck/hermes_resnet_18/pyTorch/lightning/2/reduced_dataset_10ep.ckpt

BATCH_SIZE = 64

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

In [ ]:
# Data transforms
eval_transform = transforms.Compose([
    transforms.ToTensor(), # here they become [0, 1] due to torch backend with Pillow
])

In [ ]:
# Load dataset
print("Loading test dataset...")
test_dataset = PCAMDataset(
    input_file_path=os.path.join(DATA_ROOT, "pcam/test_split.h5"),
    label_file_path=os.path.join(DATA_ROOT, "Labels/Labels/camelyonpatch_level_2_split_test_y.h5"),
    transform=eval_transform,
    max_samples=5000  # Reduced for faster evaluation
)

In [ ]:
# Calculate clip values for normalized data 
min_val = np.min([(0.0 - m) / s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)])
max_val = np.max([(1.0 - m) / s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)])
CLIP_VALUES_NORMALIZED = (min_val, max_val)
print(f"Clip values: {CLIP_VALUES_NORMALIZED}")

In [ ]:
# Load model
print("Loading model...")
model = ResNet18Classifier.load_from_checkpoint(os.path.join(MODEL_PATH, "reduced_dataset_10ep.ckpt"))

In [ ]:
@dataclass
class ExperimentResult:
    """Data class to represent a single experiment result."""
    attack: str
    defense: str
    metric: float
    details: Optional[Dict] = None

class ExperimentState:
    """Manages the state of experiments to avoid recomputation."""
    
    def __init__(self):
        self.adversarial_datasets: Dict[str, Dataset] = {}
        self.prediction_data: Dict[str, Tuple[float, Dict]] = {}
        self.detailed_results: Dict[str, Dict] = {}
    
    def has_adversarial_dataset(self, attack_name: str) -> bool:
        return attack_name in self.adversarial_datasets
    
    def get_adversarial_dataset(self, attack_name: str) -> Optional[Dataset]:
        return self.adversarial_datasets.get(attack_name)
    
    def store_adversarial_dataset(self, attack_name: str, dataset: Dataset):
        self.adversarial_datasets[attack_name] = dataset
    
    def has_predictions(self, key: str) -> bool:
        return key in self.prediction_data
    
    def get_predictions(self, key: str) -> Optional[Tuple[float, Dict]]:
        return self.prediction_data.get(key)
    
    def store_predictions(self, key: str, metric: float, details: Dict):
        self.prediction_data[key] = (metric, details)

In [ ]:
class ExperimentRunner:
    """Handles the execution of individual experiments."""
    
    def __init__(self, framework: 'MyFramework', state: ExperimentState):
        self.framework = framework
        self.state = state
    
    def run_clean_experiment(self) -> ExperimentResult:
        """Run experiment on clean data."""
        key = "clean"
        if self.state.has_predictions(key):
            metric, details = self.state.get_predictions(key)
            print(f"Using cached results for {key}")
        else:
            print(f"Computing results for {key}")
            metric, details = self.framework._evaluate_dataset(
                self.framework.test_dataset, 
                self.framework.base_classifier, 
                name="clean"
            )
            self.state.store_predictions(key, metric, details)
        
        return ExperimentResult(attack="none", defense="none", metric=metric, details=details)
    
    def run_attack_experiment(self, attack_name: str, attack_fn: Callable) -> List[ExperimentResult]:
        """Run experiments for a single attack with all defenses."""
        results = []
        
        # Generate or retrieve adversarial dataset
        dset = self._get_or_generate_adversarial_dataset(attack_name, attack_fn)
        
        # Test attack without defense
        no_defense_result = self._evaluate_attack_without_defense(attack_name, dset)
        results.append(no_defense_result)
        
        # Test attack with each defense
        defense_results = self._evaluate_attack_with_defenses(attack_name, dset)
        results.extend(defense_results)
        
        return results
    
    def _get_or_generate_adversarial_dataset(self, attack_name: str, attack_fn: Callable) -> Dataset:
        """Get existing adversarial dataset or generate a new one."""
        if self.state.has_adversarial_dataset(attack_name):
            print(f"Using cached adversarial dataset for {attack_name}")
            return self.state.get_adversarial_dataset(attack_name)
        
        print(f"Generating adversarial dataset for {attack_name}")
        dset, details = self.framework._generate_adversarial_dataset(
            self.framework.base_classifier, attack_fn, attack_name
        )
        self.state.store_adversarial_dataset(attack_name, dset)
        return dset
    
    def _evaluate_attack_without_defense(self, attack_name: str, dset: Dataset) -> ExperimentResult:
        """Evaluate attack without any defense."""
        key = f"{attack_name}_no_defense"
        if self.state.has_predictions(key):
            metric, details = self.state.get_predictions(key)
            print(f"Using cached results for {key}")
        else:
            print(f"Computing results for {key}")
            metric, details = self.framework._evaluate_dataset(
                dset, self.framework.base_classifier, name=key
            )
            self.state.store_predictions(key, metric, details)
        
        return ExperimentResult(attack=attack_name, defense="none", metric=metric, details=details)
    
    def _evaluate_attack_with_defenses(self, attack_name: str, dset: Dataset) -> List[ExperimentResult]:
        """Evaluate attack with all registered defenses."""
        results = []
        
        for defense_name, defense_fn in self.framework.defenses:
            result = self._evaluate_single_defense(attack_name, defense_name, defense_fn, dset)
            results.append(result)
        
        return results
    
    def _evaluate_single_defense(self, attack_name: str, defense_name: str, defense_fn: Callable, dset: Dataset) -> ExperimentResult:
        """Evaluate a single defense against an attack."""
        key = f"{attack_name}_{defense_name}"
        
        if self.state.has_predictions(key):
            metric, details = self.state.get_predictions(key)
            print(f"Using cached results for {key}")
            return ExperimentResult(attack=attack_name, defense=defense_name, metric=metric, details=details)
        
        print(f"Computing results for {key}")
        
        # Build classifier with defense
        classifier_with_defense = self.framework._build_classifier(defence=defense_fn)
        
        # Evaluate
        metric, details = self.framework._evaluate_dataset(dset, classifier_with_defense, name=key)
        
        self.state.store_predictions(key, metric, details)
        return ExperimentResult(attack=attack_name, defense=defense_name, metric=metric, details=details)

In [ ]:
class ResultsManager:
    """Handles saving and loading of experiment results."""
    
    @staticmethod
    def save_results(results: List[ExperimentResult], output_file: str) -> pd.DataFrame:
        """Save results to CSV file."""
        output_path = Path(output_file)
        if output_path.suffix != '.csv':
            raise ValueError("Output file must be a CSV file.")
        
        # Convert results to DataFrame
        data = [
            {"attack": r.attack, "defense": r.defense, "metric": r.metric}
            for r in results
        ]
        df = pd.DataFrame(data)
        
        # Save to file
        df.to_csv(str(output_path.absolute()), index=False, float_format='%.4f')
        print(f"Results saved to {output_path.absolute()}")
        
        return df

In [ ]:
class GaussianSmoothingGPU(art.defences.preprocessor.preprocessor.Preprocessor):
    params = ["kernel_size", "sigma", "visualize"]

    def __init__(
        self,
        kernel_size: list[int],
        sigma: Optional[list[float]] = None,
        apply_fit: bool = True,
        apply_predict: bool = True,
        visualize: bool = False
    ):
        super().__init__(is_fitted=True, apply_fit=apply_fit, apply_predict=apply_predict)
        self.kernel_size = kernel_size
        self.sigma = sigma
        self.visualize = visualize
        self._check_params()

    def __call__(self, x: np.ndarray, y: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray | None]:

        """
        Applies Gaussian smoothing to the input `x`.
        """
        # Convert to tensor if not already
        if not isinstance(x, torch.Tensor):
            x = torch.from_numpy(x)

        x_smoothed = gaussian_blur(x, kernel_size=self.kernel_size, sigma=self.sigma)

        if self.visualize:
            # Visualize the first image in the batch
            if x_smoothed.ndim == 4:
                plt.figure(figsize=(6, 6))
                plt.subplot(1, 2, 1)
                img = x[0].permute(1, 2, 0).numpy()
                plt.imshow(img)
                plt.axis('off')
                plt.title("Original Image")
                plt.subplot(1, 2, 2)
                img = x_smoothed[0].permute(1, 2, 0).numpy()
                plt.imshow(img)
                plt.axis('off')
                plt.title("Smoothed Image")
                plt.show()
                
        return x_smoothed.numpy().astype(ART_NUMPY_DTYPE), y
        
    def _check_params(self) -> None:
        if not isinstance(self.kernel_size, list) or len(self.kernel_size) != 2:
            raise ValueError("Kernel size must be a list of two integers [height, width].")
        if not all(isinstance(k, int) and k > 0 for k in self.kernel_size):
            raise ValueError("Kernel size values must be positive integers.")
        if self.sigma is not None and (not isinstance(self.sigma, list) or len(self.sigma) != 2):
            raise ValueError("Sigma must be a list of two floats [sigma_h, sigma_w].")
        if self.sigma is not None and not all(isinstance(s, (float, int)) for s in self.sigma):
            raise ValueError("Sigma values must be floats or integers.")

In [ ]:
class JpegCompression(art.defences.preprocessor.preprocessor.Preprocessor):
    params = ["quality", "channels_first", "clip_values", "verbose"]

    def __init__(
        self,
        clip_values: "CLIP_VALUES_TYPE", # Should be (0,1) for JPEG part
        quality: int = 75,
        channels_first: bool = False,
        apply_fit: bool = True,
        apply_predict: bool = True,
        verbose: bool = False,
    ):
        super().__init__(is_fitted=True, apply_fit=apply_fit, apply_predict=apply_predict)
        self.quality = quality
        self.channels_first = channels_first
        self.clip_values = clip_values # Expected (0,1) for JPEG input
        self.verbose = verbose
        self._check_params()

    def _compress(self, x: np.ndarray, mode: str) -> np.ndarray:
        from PIL import Image
        from io import BytesIO

        tmp_jpeg = BytesIO()
        x_image = Image.fromarray(x, mode=mode)
        x_image.save(tmp_jpeg, format="jpeg", quality=self.quality)
        x_jpeg = np.array(Image.open(tmp_jpeg))
        tmp_jpeg.close()
        return x_jpeg

    def __call__(self, x: np.ndarray, y: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray | None]:
        # DISCLAIMER: The following clipping is necessary to ensure that the input to the JPEG compression
        # defence is in the [0, 1] range, as required by the JPEG compression algorithm. However, this introduction of clipping
        # may not be ideal for all use cases and introduces an approxximation error. We leave it to the user to decide whether this is acceptable.
        x = np.clip(x, self.clip_values[0], self.clip_values[1])  # Clip to [0, 1] range

        x_ndim = x.ndim
        if x_ndim not in [4, 5]:
            raise ValueError(
                "Unrecognized input dimension. JPEG compression can only be applied to image and video data."
            )

        if x.min() < 0.0:
            raise ValueError(
                "Negative values in input `x` detected. The JPEG compression defence requires unnormalized input."
            )

        # Swap channel index
        if self.channels_first and x_ndim == 4:
            # image shape NCHW to NHWC
            x = np.transpose(x, (0, 2, 3, 1))
        elif self.channels_first and x_ndim == 5:
            # video shape NCFHW to NFHWC
            x = np.transpose(x, (0, 2, 3, 4, 1))

        # insert temporal dimension to image data
        if x_ndim == 4:
            x = np.expand_dims(x, axis=1)

        # Convert into uint8
        if self.clip_values[1] == 1.0:
            x = x * 255
        x = x.astype("uint8")

        # Compress one image at a time
        x_jpeg = x.copy()
        for idx in np.ndindex(x.shape[:2]):
            if x.shape[-1] == 3:
                x_jpeg[idx] = self._compress(x[idx], mode="RGB")
            else:
                for i_channel in range(x.shape[-1]):
                    x_channel = x[idx[0], idx[1], ..., i_channel]
                    x_channel = self._compress(x_channel, mode="L")
                    x_jpeg[idx[0], idx[1], :, :, i_channel] = x_channel

        # Convert to ART dtype
        if self.clip_values[1] == 1.0:
            x_jpeg = x_jpeg / 255.0
        x_jpeg = x_jpeg.astype(ART_NUMPY_DTYPE)

        # remove temporal dimension for image data
        if x_ndim == 4:
            x_jpeg = np.squeeze(x_jpeg, axis=1)

        # Swap channel index
        if self.channels_first and x_jpeg.ndim == 4:
            # image shape NHWC to NCHW
            x_jpeg = np.transpose(x_jpeg, (0, 3, 1, 2))
        elif self.channels_first and x_ndim == 5:
            # video shape NFHWC to NCFHW
            x_jpeg = np.transpose(x_jpeg, (0, 4, 1, 2, 3))
        
        return x_jpeg, y

    def _check_params(self) -> None:
        if not isinstance(self.quality, int) or self.quality <= 0 or self.quality > 100:
            raise ValueError("Image quality must be a positive integer <= 100.")

        if len(self.clip_values) != 2:
            raise ValueError("'clip_values' should be a tuple of 2 floats or arrays containing the allowed data range.")

        if np.array(self.clip_values[0] >= self.clip_values[1]).any():
            raise ValueError("Invalid 'clip_values': min >= max.")

        if self.clip_values[0] != 0:
            raise ValueError("'clip_values' min value must be 0.")

        if self.clip_values[1] != 1.0 and self.clip_values[1] != 255:
            raise ValueError("'clip_values' max value must be either 1 or 255.")

        if not isinstance(self.verbose, bool):
            raise ValueError("The argument `verbose` has to be of type bool.")

In [ ]:
class TotalVarMininimization(art.defences.preprocessor.preprocessor.Preprocessor):
    params = ["prob", "norm", "lamb", "solver", "max_iter", "clip_values", "verbose"]

    def __init__(
        self,
        prob: float = 0.3,
        norm: int = 2,
        lamb: float = 0.5,
        solver: str = "L-BFGS-B",
        max_iter: int = 10,
        clip_values: "CLIP_VALUES_TYPE" | None = None,
        apply_fit: bool = False,
        apply_predict: bool = True,
        verbose: bool = False,
    ):
        """
        Create an instance of total variance minimization.

        :param prob: Probability of the Bernoulli distribution.
        :param norm: The norm (positive integer).
        :param lamb: The lambda parameter in the objective function.
        :param solver: Current support: `L-BFGS-B`, `CG`, `Newton-CG`.
        :param max_iter: Maximum number of iterations when performing optimization.
        :param clip_values: Tuple of the form `(min, max)` representing the minimum and maximum values allowed
               for features.
        :param apply_fit: True if applied during fitting/training.
        :param apply_predict: True if applied during predicting.
        :param verbose: Show progress bars.
        """
        super().__init__(is_fitted=True, apply_fit=apply_fit, apply_predict=apply_predict)
        self.prob = prob
        self.norm = norm
        self.lamb = lamb
        self.solver = solver
        self.max_iter = max_iter
        self.clip_values = clip_values
        self.verbose = verbose
        self._check_params()

    def __call__(self, x: np.ndarray, y: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray | None]:
        """
        Apply total variance minimization to sample `x`.

        :param x: Sample to compress with shape `(batch_size, width, height, depth)`.
        :param y: Labels of the sample `x`. This function does not affect them in any way.
        :return: Similar samples.
        """
        if len(x.shape) == 2:
            raise ValueError(
                "Feature vectors detected. Variance minimization can only be applied to data with spatial dimensions."
            )
        x_preproc = x.copy()

        def _optimize_channel(x_i, channel_idx):
            """Optimize a single channel of an image."""
            mask = (np.random.rand(*x_i.shape) < self.prob).astype("int")
            res = minimize(
                self._loss_func,
                x_i[:, :, channel_idx].flatten(),
                (x_i[:, :, channel_idx], mask[:, :, channel_idx], self.norm, self.lamb),
                method=self.solver,
                jac=self._deri_loss_func,
                options={"maxiter": self.max_iter},
            )
            return channel_idx, np.reshape(res.x, x_i[:, :, channel_idx].shape)

        def _process_image(i, x_i):
            """Process a single image with parallel channel optimization."""
            z_min = x_i.copy()
            
            # Parallelize across channels
            results = Parallel(n_jobs=-1)(
                delayed(_optimize_channel)(x_i, channel_idx) 
                for channel_idx in range(x_i.shape[2])
            )
            
            # Update channels with optimized results
            for channel_idx, optimized_channel in results:
                z_min[:, :, channel_idx] = optimized_channel
                
            return i, z_min

        # Parallelize across batch items
        batch_results = Parallel(n_jobs=-1)(
            delayed(_process_image)(i, x_i) 
            for i, x_i in enumerate(tqdm(x_preproc, desc="Variance minimization", disable=not self.verbose))
        )
        
        # Update the preprocessed data
        for i, z_min in batch_results:
            x_preproc[i] = z_min

        if self.clip_values is not None:
            np.clip(x_preproc, self.clip_values[0], self.clip_values[1], out=x_preproc)

        return x_preproc.astype(ART_NUMPY_DTYPE), y

    @staticmethod
    def _loss_func(z_init: np.ndarray, x: np.ndarray, mask: np.ndarray, norm: int, lamb: float) -> float:
        """
        Loss function to be minimized.

        :param z_init: Initial guess.
        :param x: Original image.
        :param mask: A matrix that decides which points are kept.
        :param norm: The norm (positive integer).
        :param lamb: The lambda parameter in the objective function.
        :return: Loss value.
        """
        res = np.sqrt(np.power(z_init - x.flatten(), 2).dot(mask.flatten()))
        z_init = np.reshape(z_init, x.shape)
        res += lamb * np.linalg.norm(z_init[1:, :] - z_init[:-1, :], norm, axis=1).sum()
        res += lamb * np.linalg.norm(z_init[:, 1:] - z_init[:, :-1], norm, axis=0).sum()

        return res

    @staticmethod
    def _deri_loss_func(z_init: np.ndarray, x: np.ndarray, mask: np.ndarray, norm: int, lamb: float) -> float:
        """
        Derivative of loss function to be minimized.

        :param z_init: Initial guess.
        :param x: Original image.
        :param mask: A matrix that decides which points are kept.
        :param norm: The norm (positive integer).
        :param lamb: The lambda parameter in the objective function.
        :return: Derivative value.
        """
        # First compute the derivative of the first component of the loss function
        nor1 = np.sqrt(np.power(z_init - x.flatten(), 2).dot(mask.flatten()))
        nor1 = max(nor1, 1e-06)
        der1 = ((z_init - x.flatten()) * mask.flatten()) / (nor1 * 1.0)

        # Then compute the derivative of the second component of the loss function
        z_init = np.reshape(z_init, x.shape)

        if norm == 1:
            z_d1 = np.sign(z_init[1:, :] - z_init[:-1, :])
            z_d2 = np.sign(z_init[:, 1:] - z_init[:, :-1])
        else:
            z_d1_norm = np.power(np.linalg.norm(z_init[1:, :] - z_init[:-1, :], norm, axis=1), norm - 1)
            z_d2_norm = np.power(np.linalg.norm(z_init[:, 1:] - z_init[:, :-1], norm, axis=0), norm - 1)
            z_d1_norm[z_d1_norm < 1e-6] = 1e-6
            z_d2_norm[z_d2_norm < 1e-6] = 1e-6
            z_d1_norm = np.repeat(z_d1_norm[:, np.newaxis], z_init.shape[1], axis=1)
            z_d2_norm = np.repeat(z_d2_norm[np.newaxis, :], z_init.shape[0], axis=0)
            z_d1 = norm * np.power(z_init[1:, :] - z_init[:-1, :], norm - 1) / z_d1_norm
            z_d2 = norm * np.power(z_init[:, 1:] - z_init[:, :-1], norm - 1) / z_d2_norm

        der2 = np.zeros(z_init.shape)
        der2[:-1, :] -= z_d1
        der2[1:, :] += z_d1
        der2[:, :-1] -= z_d2
        der2[:, 1:] += z_d2
        der2 = lamb * der2.flatten()

        # Total derivative
        return der1 + der2

    def _check_params(self) -> None:
        if not isinstance(self.prob, (float, int)) or self.prob < 0.0 or self.prob > 1.0:
            raise ValueError("Probability must be between 0 and 1.")

        if not isinstance(self.norm, int) or self.norm <= 0:
            raise ValueError("Norm must be a positive integer.")

        if self.solver not in ("L-BFGS-B", "CG", "Newton-CG"):
            raise ValueError("Current support only L-BFGS-B, CG, Newton-CG.")

        if not isinstance(self.max_iter, int) or self.max_iter <= 0:
            raise ValueError("Number of iterations must be a positive integer.")

        if self.clip_values is not None:

            if len(self.clip_values) != 2:
                raise ValueError("`clip_values` should be a tuple of 2 floats containing the allowed data range.")

            if np.array(self.clip_values[0] >= self.clip_values[1]).any():
                raise ValueError("Invalid `clip_values`: min >= max.")

        if not isinstance(self.verbose, bool):
            raise ValueError("The argument `verbose` has to be of type bool.")

In [ ]:
@dataclass
class MyFramework:
    """
    Modular framework for evaluating ART attacks and defenses witha simple OOP design.
    """
    model: torch.nn.Module
    test_dataset: torch.utils.data.Dataset
    loss: torch.nn.Module
    optimizer: torch.optim.Optimizer
    input_shape: Tuple[int, int, int]
    num_classes: int
    channels_first: bool
    clip_values: Optional[Tuple[int, int]] = None
    preprocessing: Optional[Tuple[np.ndarray, np.ndarray]] = None
    batch_size: int = 64
    metric: Optional[str] = 'accuracy'
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    def __post_init__(self):
        self.class_names = [f"Class_{i}" for i in range(self.num_classes)]

        # Instantiate base classifier without defenses
        self.base_classifier = self._build_classifier(defence=None)

        # Registries for attacks and defenses
        self.attacks: List[Tuple[str, Callable]] = []
        self.defenses: List[Tuple[str, Callable]] = []

        # State management
        self.state = ExperimentState()
        
        # Experiment runner
        self.runner = ExperimentRunner(self, self.state)

    def _build_classifier(self, defence: Optional[Any] = None):
        """Return a new PyTorchClassifier with optional preprocessing defenses."""
            
        return PyTorchClassifier(
            model=self.model,
            loss=self.loss,
            optimizer=self.optimizer,
            input_shape=self.input_shape,
            nb_classes=self.num_classes,
            channels_first=self.channels_first,
            clip_values=self.clip_values,
            preprocessing_defences=defence if defence else None,
            preprocessing=self.preprocessing,
            device_type=self.device,
        )

    def _make_dataloader(self, dataset: torch.utils.data.Dataset) -> DataLoader:
        """Create a DataLoader for the given dataset."""
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=False,
            pin_memory=True,
        )

    def _generate_adversarial_dataset(
        self,
        classifier,
        attack_fn: Callable,
        attack_name: str
    ) -> Tuple[Dataset, Dict]:
        """Generate adversarial examples and collect detailed metrics."""
        attack = attack_fn(classifier)
        loader = self._make_dataloader(self.test_dataset)

        adv_images, adv_labels = [], []
        original_images, perturbations = [], []
        original_preds, adv_preds = [], []
        original_confs, adv_confs = [], []

        for X, y in tqdm(loader, desc=f"Crafting adversarial ({attack.__class__.__name__})"):
            # Generate adversarial examples
            x_adv = attack.generate(x=X.numpy(), y=y)
            
            # Get predictions and confidences
            orig_pred_probs = classifier.predict(X.numpy(), batch_size=self.batch_size, training_mode = False)
            adv_pred_probs = classifier.predict(x_adv, batch_size=self.batch_size, training_mode = False)
            
            # Store data
            adv_images.append(x_adv)
            adv_labels.append(y)
            original_images.append(X.numpy())
            perturbations.append(x_adv - X.numpy())
            
            original_preds.append(orig_pred_probs.argmax(axis=1))
            adv_preds.append(adv_pred_probs.argmax(axis=1))
            original_confs.append(orig_pred_probs.max(axis=1))
            adv_confs.append(adv_pred_probs.max(axis=1))

        # Concatenate all data
        X_adv = np.concatenate(adv_images, axis=0)
        y_adv = torch.cat(adv_labels)
        X_orig = np.concatenate(original_images, axis=0)
        perturbations_all = np.concatenate(perturbations, axis=0)
        
        # Create detailed results dictionary
        details = {
            'original_images': X_orig,
            'adversarial_images': X_adv,
            'perturbations': perturbations_all,
            'original_predictions': np.concatenate(original_preds),
            'adversarial_predictions': np.concatenate(adv_preds),
            'original_confidences': np.concatenate(original_confs),
            'adversarial_confidences': np.concatenate(adv_confs),
            'true_labels': y_adv.numpy()
        }
        
        self.state.detailed_results[attack_name] = details
        
        return TensorDataset(torch.from_numpy(X_adv), y_adv), details

    def _evaluate_dataset(
        self,
        dataset: torch.utils.data.Dataset,
        classifier,
        name: str
    ) -> Tuple[float, Dict]:
        """Compute metric and collect detailed prediction information."""
        loader = self._make_dataloader(dataset)
        all_probs, all_preds, all_labels, all_confs = [], [], [], []
        
        for X, y in tqdm(loader, desc=f"Evaluating {name} dataset"):
            pred_probs = classifier.predict(X.numpy(), batch_size=self.batch_size, training_mode=False)
            preds = pred_probs.argmax(axis=1)
            confs = pred_probs.max(axis=1)
            
            all_probs.append(pred_probs)
            all_preds.extend(preds)
            all_labels.extend(y.numpy())
            all_confs.extend(confs)

        full_probs = np.concatenate(all_probs, axis=0)
        
        if self.metric == 'accuracy':
            metric = accuracy_score(all_labels, all_preds)
        elif self.metric == 'f1_score':
            metric = f1_score(all_labels, all_preds)
        elif self.metric == 'AUROC':
            metric = roc_auc_score(all_labels, full_probs[:, 1])
        else:
            raise NotImplementedError(f'metric {self.metric} is not implemented')
        
        details = {
            'predictions': np.array(all_preds),
            'true_labels': np.array(all_labels),
            'confidences': np.array(all_confs)
        }
        
        return metric, details

    def add_attack(self, name: str, attack_fn: Callable):
        """Register an attack factory."""
        if any(name == atk_name for atk_name, _ in self.attacks):
            print(f"Attack '{name}' is already registered. Skipping.")
            return
        
        if not callable(attack_fn):
            raise ValueError(f"Attack '{name}' must be a callable function.")
        
        self.attacks.append((name, attack_fn))

    def add_defense(self, name: str, defense_fn: Callable):
        """Register a defense factory."""
        if any(name == def_name for def_name, _ in self.defenses):
            print(f"Defense '{name}' is already registered. Skipping.")
            return
        
        if not callable(defense_fn):
            raise ValueError(f"Defense '{name}' must be a callable function.")
        
        self.defenses.append((name, defense_fn))

    def run_experiments(self, output_file: str) -> pd.DataFrame:
        """
        Run all combinations of clean, attacks, and defenses.
        Returns results DataFrame.
        """
        all_results = []
        
        # Run clean experiment
        print("Running clean experiment...")
        clean_result = self.runner.run_clean_experiment()
        all_results.append(clean_result)
        
        # Run attack experiments
        for attack_name, attack_fn in self.attacks:
            print(f"\nProcessing attack: {attack_name}")
            attack_results = self.runner.run_attack_experiment(attack_name, attack_fn)
            all_results.extend(attack_results)

        # Run defenses in isolation
        for defense_name, defense_fn in self.defenses:
            print(f"\nProcessing defense: {defense_name}")
            defense_results = self.runner._evaluate_single_defense(attack_name="none", defense_name=defense_name, defense_fn=defense_fn, dset=self.test_dataset)
            all_results.append(defense_results)
        
        # Save and return results
        return ResultsManager.save_results(all_results, output_file)

    #####################
    #        PLOT       #
    #####################
    def plot_example(self, example: torch.Tensor, original: Optional[torch.Tensor] = None):
        """Plot an example tensor and showing original image."""
            
        # Rearrange and convert the example tensor to a NumPy array
        img = rearrange(example, 'C H W -> H W C').numpy().squeeze()
    
        if original is not None:
            # Rearrange and convert the original tensor to a NumPy array
            orig_img = rearrange(original, 'C H W -> H W C').numpy().squeeze()
    
            # Create a figure with two subplots side by side
            fig, axes = plt.subplots(1, 2)
    
            # Display the original image
            axes[0].imshow(orig_img)
            axes[0].set_title('Original')
            axes[0].axis('off')
    
            # Display the processed image
            axes[1].imshow(img)
            axes[1].set_title('Processed')
            axes[1].axis('off')
    
            plt.tight_layout()
            plt.show()
        else:
            # Display only the processed image
            fig = plt.figure(figsize=(10, 10))
            ax = plt.Axes(fig, [0., 0., 1., 1.])
            ax.set_axis_off()
            fig.add_axes(ax)
            plt.imshow(img)
            plt.axis('off')
            #plt.title('Processed')
            plt.savefig("example_image.svg", format='svg', dpi=300)
            plt.savefig("example_image.png", format='png', dpi=300)
            plt.savefig("example_image.pdf", format='pdf', dpi=300)
            plt.show()

    @property
    def detailed_results(self) -> Dict:
        """Access to detailed results for backward compatibility."""
        return self.state.detailed_results
    
    @property
    def adversarial_datasets(self) -> Dict:
        """Access to adversarial datasets for backward compatibility."""
        return self.state.adversarial_datasets
    
    @property
    def prediction_data(self) -> Dict:
        """Access to prediction data for backward compatibility."""
        return self.state.prediction_data

In [ ]:
frame = MyFramework(
    model=model.model,
    test_dataset=test_dataset,
    loss=nn.CrossEntropyLoss(),
    optimizer=optim.AdamW(model.model.parameters(), lr=1e-3),
    input_shape=(3, 96, 96),
    num_classes=2,
    channels_first=True,
    clip_values=CLIP_VALUES_NORMALIZED,
    preprocessing=(IMAGENET_MEAN, IMAGENET_STD),
    batch_size=BATCH_SIZE,
    metric='AUROC'
)

Run evaluation with the base classifier on the clean test dataset. Pass the classifier to use

In [ ]:
frame._evaluate_dataset(dataset=test_dataset, classifier=frame.base_classifier, name="clean")

Craft some adverarial examples (still need to modularize the kind of attacks and parameters). Pass the dataset to use and the classifier to use

In [ ]:
adversarial_dataset, _ = frame._generate_adversarial_dataset(
    classifier=frame.base_classifier,
    attack_fn=lambda clf: FastGradientMethod(
        estimator=clf,
        eps=0.02,
    ),
    attack_name="FGSM",
)

Visualize the adversarial example and the original one or just one of them if original is not specified

In [ ]:
idx = 1
frame.plot_example(example=adversarial_dataset[idx][0], original=test_dataset[idx][0])

Run evaluation of the base classifier on the adversarial dataset given by applying the attack over the clean test dataset

In [ ]:
frame._evaluate_dataset(dataset=adversarial_dataset, classifier=frame.base_classifier, name="FGSM_no_defense")

Now, create another classifier which incorporates one or more defense strategies

In [ ]:
defense_classifier = frame._build_classifier(defence=GaussianSmoothingGPU(kernel_size=[3,3], sigma=[0.4, 0.4]))

Run evaluation on the dataset with adversarial examples using the defense classifier rather than the base classifier. The defense classifier has inside some defences

In [ ]:
frame._evaluate_dataset(dataset=adversarial_dataset, classifier=defense_classifier, name="FGSM_gaussian_smoothing")

Now, try the full pipeline. First instantiate a clean framework and then add attacks and defenses

---

# Main Pipeline

The framework is designed to run experiments on adversarial attacks and defenses. Our idea is to create a modular framework that allows users to easily add new attacks and defenses, and run experiments on them.

## General idea

The framework instantiates a base classifier, which is a ResNet18 model trained on the ImageNet dataset. It then evaluates the classifier on a clean test dataset. 
After that, depending on the attacks and defenses defined, it generates adversarial examples using the base classifier and evaluates them. 
Finally, it evaluates the adversarial examples using a defense classifier that incorporates one or more defense strategies.

## Running experiments

The run experiments works in this way:

- creates an empty object to store the metrics
- evaluates the clean test dataset with the base classifier
- loops over the defined attacks (you need to add them, check above)
- for each attack it generates a torch dataset applying the attack on each sample and then evaluates the adversarial crafted test set with base classifier
- then it loops over the defenses and for each it builds the classifier using only that defense and then it evaluates the adversarial crafted test set with the classifier having that defense as countermeasure.

## Statefulness

The framework is stateful, meaning that it keeps track of the current state of the experiments. Once an experiment is run, the metrics are stored in the framework object as well as the adversarial examples generated. In this way, running again the same experiment will not generate new adversarial examples, but will use the ones already generated. Moreover, when evaluating the performance of the classifier, if the results are already stored in the framework, it will not run the evaluation again, but will use the stored results, pushing them on the results dictionary which is then converted to a pandas dataframe.

## Notes
- >**Info**: The framework is designed to use only preprocessor defenses, which are applied to the input data before passing it to the classifier.

In [33]:
clean_frame = MyFramework(
    model=model.model,
    test_dataset=test_dataset,
    loss=nn.CrossEntropyLoss(),
    optimizer=optim.AdamW(model.model.parameters(), lr=1e-3),
    input_shape=(3, 96, 96),
    num_classes=2,
    channels_first=True,
    batch_size=BATCH_SIZE,
    clip_values=CLIP_VALUES_NORMALIZED,
    preprocessing=(IMAGENET_MEAN, IMAGENET_STD),
    metric='AUROC'
)

Add attacks

In [34]:
for eps in np.arange(0.01, 0.11, 0.01):
    eps = round(eps, 2)
    clean_frame.add_attack(f"PGD (eps={eps})", lambda clf, eps=eps: ProjectedGradientDescent(estimator=clf, eps=eps, max_iter=30))
    #clean_frame.add_attack(f"FGSM (eps={eps})", lambda clf, eps=eps: FastGradientMethod(estimator=clf, eps=eps))

Add defenses

In [35]:
clean_frame.add_defense("GaussianSmoothing (k=3, sigma=0.2)", GaussianSmoothingGPU(kernel_size=[3,3], sigma=[0.2, 0.2]))
clean_frame.add_defense("GaussianSmoothing (k=3, sigma=0.4)", GaussianSmoothingGPU(kernel_size=[3,3], sigma=[0.4, 0.4]))
clean_frame.add_defense("GaussianSmoothing (k=3, sigma=0.6)", GaussianSmoothingGPU(kernel_size=[3,3], sigma=[0.6, 0.6]))
clean_frame.add_defense("GaussianSmoothing (k=3, sigma=0.8)", GaussianSmoothingGPU(kernel_size=[3,3], sigma=[0.8, 0.8]))


clean_frame.add_defense("GaussianSmoothing (k=5, sigma=0.2)", GaussianSmoothingGPU(kernel_size=[5,5], sigma=[0.2, 0.2]))
clean_frame.add_defense("GaussianSmoothing (k=5, sigma=0.4)", GaussianSmoothingGPU(kernel_size=[5,5], sigma=[0.4, 0.4]))
clean_frame.add_defense("GaussianSmoothing (k=5, sigma=0.6)", GaussianSmoothingGPU(kernel_size=[5,5], sigma=[0.6, 0.6]))
clean_frame.add_defense("GaussianSmoothing (k=5, sigma=0.8)", GaussianSmoothingGPU(kernel_size=[5,5], sigma=[0.8, 0.8]))


clean_frame.add_defense("JpegCompression (q=50)", JpegCompression(clip_values=(0,1), quality=50, channels_first=True))
clean_frame.add_defense("JpegCompression (q=70)", JpegCompression(clip_values=(0,1), quality=70, channels_first=True))
clean_frame.add_defense("JpegCompression (q=90)", JpegCompression(clip_values=(0,1), quality=90, channels_first=True))


clean_frame.add_defense("SpatialSmoothing (w=3)", SpatialSmoothingPyTorch(window_size=3, channels_first=True))
clean_frame.add_defense("SpatialSmoothing (w=5)", SpatialSmoothingPyTorch(window_size=5, channels_first=True))

clean_frame.add_defense("Total Variance Minimization (p=0.3)", TotalVarMininimization(prob=0.3, max_iter=10))

In [36]:
results_df = clean_frame.run_experiments(output_file="results_pgd_aug.csv")

Running clean experiment...
Computing results for clean


Evaluating clean dataset:   0%|          | 0/79 [00:00<?, ?it/s]


Processing attack: PGD (eps=0.01)
Generating adversarial dataset for PGD (eps=0.01)


Crafting adversarial (ProjectedGradientDescent):   0%|          | 0/79 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...


Processing defense: SpatialSmoothing (w=5)
Computing results for none_SpatialSmoothing (w=5)


Evaluating none_SpatialSmoothing (w=5) dataset:   0%|          | 0/79 [00:00<?, ?it/s]


Processing defense: Total Variance Minimization (p=0.3)
Computing results for none_Total Variance Minimization (p=0.3)


Evaluating none_Total Variance Minimization (p=0.3) dataset:   0%|          | 0/79 [00:00<?, ?it/s]

Results saved to /users/mfasulo/HERMES/notebooks/results_pgd_aug.csv


In [37]:
results_df.sort_values(by="metric", ascending=False)

,attack,defense,metric
160,none,JpegCompression (q=70),0.942072
161,none,JpegCompression (q=90),0.941847
155,none,"GaussianSmoothing (k=5, sigma=0.2)",0.939952
151,none,"GaussianSmoothing (k=3, sigma=0.2)",0.939951
0,none,none,0.939950
...,...,...,...
142,PGD (eps=0.1),"GaussianSmoothing (k=5, sigma=0.4)",0.000000
141,PGD (eps=0.1),"GaussianSmoothing (k=5, sigma=0.2)",0.000000
108,PGD (eps=0.08),"GaussianSmoothing (k=3, sigma=0.4)",0.000000
138,PGD (eps=0.1),"GaussianSmoothing (k=3, sigma=0.4)",0.000000
